# 🎯 [C202_M05] 네트워크 혼잡도 기반 전송 성공 확률 및 링크 신뢰도 산출 엔진 개발
### Google Colab 실습 환경 (지휘통제(C2) / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/C202/c202_m05_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 지휘통제(C2) (Level 2) |
| **미션 ID** | C202_M05 |
| **미션 주제** | 네트워크 혼잡도 기반 전송 성공 확률 및 링크 신뢰도 산출 엔진 개발 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
러시아-우크라이나 전쟁에서는 전자전 재밍과 통신 장비 부족으로 협대역 무선망의 대역폭이 크게 줄어드는 상황이 잦았고, 이때 적 접촉 보고나 의무후송 요청 같은 긴급 전문이 행정 전문 뒤에 묶여 늦게 전달되는 문제가 지휘통제의 약점이 되었다. 본 프로젝트에서는 가상 협대역 링크(2.4kbps)와 가상 전술 전문을 코드로 직접 만들어, 전문 구조 정의, 패킷 파싱·압축 전처리, 긴급도 기반 우선순위 큐, 채널 상태에 따른 지능형 재전송, 링크 신뢰도 산출, 통신 관제 화면, 통신관 HITL 통제, 대역폭 급감 스트레스 시험, AAR 재조정, 최종 통합까지 메시지 우선순위 큐·재전송 프로토타입을 단계별로 구현한다. 최종 목표는 대역폭이 제한된 상황에서도 FLASH·IMMEDIATE 전문을 기한 안에 전달하고, 어떤 전문도 잃어버리지 않으며, 순서 조정의 최종 판단은 통신관이 내리도록 설계하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 지휘통제체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/c202_input_schema.json` | 입출력 데이터 표준 스키마 및 원천/정제 데이터셋 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/verification/`, `src/ui/`, `src/hitl/`, `src/main_c202.py` | 전처리, 우선순위 큐, 재전송 제어, 링크 신뢰도, 대시보드, HITL 등 핵심 파이썬 소스 코드 |
| **`docs/`** | `docs/c202_data_spec.md`, `docs/c202_hitl_guide.md`, `docs/c202_aar_report.md`, `docs/c202_final_report.md` | 데이터 규격서, HITL 가이드, AAR 분석 보고서, 최종 브리핑 등 기술/지휘관 문서 |
| **`configs/`**| `configs/c202_tuned_params.yaml` | AAR 피드백 기반 재조정 임계치 및 환경설정 파일 |
| **`tests/`** | `tests/c202_stress_test.py` | 대역폭 90% 급감 스트레스 시험 및 단위 테스트 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (권장):** 코드 셀 첫 줄에 `%%writefile src/preprocess/c202_preprocessor.py`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 05. 네트워크 혼잡도 기반 전송 성공 확률 및 링크 신뢰도 산출 엔진 개발
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

무선 링크 품질 및 혼잡도를 반영하여 전송 성공 확률과 링크 신뢰도를 산출하는 엔진을 개발한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **링크 품질과 혼잡도를 반영해 조각 전송 성공 확률과 링크 신뢰도 등급(High/Medium/Low)을 산출하는 엔진을 개발한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **SNR, 최근 손실률, 큐 점유율, 조각 크기, 지연 변동을 특징으로 학습용·검증용 가상 데이터를 나눠 그래디언트 부스팅 엔진을 학습 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/verification/c202_confidence_engine.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 신뢰도 등급(High/Med/Low) 부여 기준의 군사적 타당성

#### 미션 PASS 조건

- 정량 기준: 오경보율(FAR) 40% 이상 감축, 신뢰도 스코어 산출 일관성 90% 이상


- 진행 기준: 오경보 필터링 전후 비교 지표 산출 및 통합 테스트 완료


- 검증 방법: ROC-AUC 곡선 분석 및 임계값 민감도 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""무선 링크 품질(SNR·최근 손실률·지연 변동)과 혼잡도(큐 점유율)·조각 크기로 전송 성공 확률과 링크 신뢰도 등급(High/Medium/Low)을 산출하는 엔진을 개발하고, 단순 규칙 대비 '링크 불량' 오경보율(FAR) 감축, ROC-AUC, 임계값 민감도, 점수 일관성을 검증한다."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, roc_curve

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 전술 전문(메시지) 생성기 (실제 부대·통신망·제원 정보 없음)
# 긴급도 4단계(FLASH/IMMEDIATE/PRIORITY/ROUTINE)와 전문 유형별 크기를 가진 전문을 만든다.
# ---------------------------------------------------------------------------
PREC = ["FLASH", "IMMEDIATE", "PRIORITY", "ROUTINE"]
RANK = {p: i for i, p in enumerate(PREC)}
DEADLINE = {"FLASH": 10, "IMMEDIATE": 30, "PRIORITY": 120, "ROUTINE": 600}   # 가상 전달 목표 시간(초)
MSG_TYPES = {   # 유형: (긴급도 후보, 확률, 본문 크기 범위 bytes, 발생 비율)
    "CONTACT": (["FLASH", "IMMEDIATE"], [0.7, 0.3], (120, 300), 0.08),
    "MEDEVAC": (["FLASH", "IMMEDIATE"], [0.4, 0.6], (150, 350), 0.05),
    "SITREP": (["IMMEDIATE", "PRIORITY"], [0.2, 0.8], (600, 2500), 0.20),
    "POSREP": (["PRIORITY", "ROUTINE"], [0.5, 0.5], (60, 120), 0.35),
    "LOGREQ": (["PRIORITY", "ROUTINE"], [0.4, 0.6], (300, 1200), 0.14),
    "ADMIN": (["ROUTINE"], [1.0], (1000, 5000), 0.18),
}


def make_messages(duration=600, rate=0.12, seed=7, t_start=0.0, prefix="M"):
    """duration초 동안 평균 rate건/초로 발생하는 가상 전문 목록."""
    rng = np.random.default_rng(seed)
    n = int(rng.poisson(rate * duration))
    times = np.sort(rng.uniform(t_start, t_start + duration, n))
    types = list(MSG_TYPES)
    msgs = []
    for i, t0 in enumerate(times):
        ty = types[rng.choice(len(types), p=[MSG_TYPES[k][3] for k in types])]
        cands, p, (lo, hi), _ = MSG_TYPES[ty]
        msgs.append({"id": f"{prefix}{i:04d}", "t0": float(t0), "type": ty, "prec": str(rng.choice(cands, p=p)),
                     "size": int(rng.integers(lo, hi + 1)), "src": f"U{int(rng.integers(1, 9)):02d}"})
    return msgs


class Link:
    """[교육용가상자료] 가상 협대역 무선 링크(기본 2.4kbps). 초 단위 상태(good/bad/jam)에 따라
    비트오류율이 바뀌고(Gilbert-Elliott 모델), 구간별 대역폭 감소도 줄 수 있다."""
    BER = {"good": 1e-5, "bad": 2e-4, "jam": 1.2e-3}

    def __init__(self, horizon=4000, bw_bps=2400, seed=1, p_gb=0.02, p_bg=0.15, jam=(), bw_drop=(), rtt=(1.2, 0.4)):
        rng = np.random.default_rng(seed)
        st, s = [], "good"
        for _ in range(int(horizon) + 1):
            s = ("bad" if rng.random() < p_gb else "good") if s == "good" else ("good" if rng.random() < p_bg else "bad")
            st.append(s)
        for a, b in jam:
            for k in range(int(a), min(int(b), len(st))):
                st[k] = "jam"
        self.states, self.bw_bps, self.bw_drop, self.rtt_par = st, bw_bps, list(bw_drop), rtt

    def state(self, t):
        return self.states[min(int(t), len(self.states) - 1)]

    def bw(self, t):
        f = 1.0
        for a, b, fac in self.bw_drop:
            if a <= t < b:
                f = fac
        return self.bw_bps * f

    def p_loss(self, nbytes, t):
        return 1 - (1 - self.BER[self.state(t)]) ** (8 * nbytes)

    def rtt(self, rng):
        return self.rtt_par[0] + rng.exponential(self.rtt_par[1])
# ---------------------------------------------------------------------------

HDR = 32
FEATS = ["snr_db", "recent_loss", "queue_util", "pkt_bytes", "rtt_jitter"]


def make_link_windows(n=6000, seed=1):
    """[교육용가상자료] 1초 창마다 링크 관측값과 그 창에서 보낸 조각 1개의 성공 여부. 실제 성공 확률(p_true)도 함께 둔다."""
    rng = np.random.default_rng(seed)
    link = Link(n, seed=seed, p_gb=0.05, p_bg=0.15, jam=[(1000, 1060), (3500, 3540), (5000, 5080)])
    snr_mu = {"good": 17, "bad": 9, "jam": 2}
    rows = []
    for k in range(1, n):
        s, prev = link.state(k), link.state(k - 1)
        size = int(rng.choice([64, 128, 256])) + HDR
        util = float(np.clip(rng.beta(2, 3) + (0.35 if 2000 <= k < 2600 else 0), 0, 1))     # 혼잡 구간 포함
        p_true = (1 - link.p_loss(size, k)) * (1 - np.clip((util - 0.85) * 4, 0, 0.6))     # 비트오류 + 큐 넘침
        rows.append({"snr_db": snr_mu[s] + rng.normal(0, 2.5), "recent_loss": rng.binomial(10, link.p_loss(288, k - 1)) / 10,
                     "queue_util": util, "pkt_bytes": size, "rtt_jitter": rng.exponential(0.3) + (0.4 if s != "good" else 0),
                     "state": s, "p_true": p_true, "ok": int(rng.random() < p_true)})
    return pd.DataFrame(rows)


def grade(p):
    return np.where(p >= 0.85, "High", np.where(p >= 0.6, "Medium", "Low"))


def far_recall(alarm, bad):
    far = float((alarm & ~bad).sum() / max(alarm.sum(), 1))
    return round(far, 3), round(float((alarm & bad).sum() / max(bad.sum(), 1)), 3), int(alarm.sum())


train, test = make_link_windows(6000, seed=1), make_link_windows(6000, seed=2)     # 학습용 / 검증용 분리
def new_model():
    return HistGradientBoostingClassifier(max_depth=3, max_iter=150, learning_rate=0.05, random_state=0)


model = new_model().fit(train[FEATS], train["ok"])
test["p"] = model.predict_proba(test[FEATS])[:, 1]
test["grade"] = grade(test["p"])
bad = (test["p_true"] < 0.6).values                       # 실제 불량 링크(정답, 시뮬레이터만 알고 있음)

auc = roc_auc_score(test["ok"], test["p"])
print(f"전송 성공 예측 ROC-AUC {auc:.3f} (검증 창 {len(test)}개, 실제 불량 링크 창 {bad.sum()}개)")

rule = ((test["snr_db"] < 12) | (test["recent_loss"] >= 0.4)).values          # 기존: 단순 규칙 경보
eng = (test["p"] < 0.6).values                                                 # 엔진: 성공 확률 0.6 미만 경보
f0, r0, n0 = far_recall(rule, bad)
f1, r1, n1 = far_recall(eng, bad)
print(pd.DataFrame([{"방식": "기존 규칙(SNR<12dB 또는 손실>=40%)", "경보": n0, "오경보율": f0, "불량 재현율": r0},
                    {"방식": "신뢰도 엔진(성공확률<0.6)", "경보": n1, "오경보율": f1, "불량 재현율": r1}]).to_string(index=False))
cut = 1 - f1 / max(f0, 1e-9)
print(f"오경보율 감축 {cut:.1%} (목표 40% 이상: {'달성' if cut >= 0.4 else '미달'})")

# 임계값 민감도
print("\n경보 임계값  경보수  오경보율  불량 재현율")
for th in [0.4, 0.5, 0.6, 0.7, 0.8]:
    f, r, n = far_recall((test["p"] < th).values, bad)
    print(f"   {th:.1f}      {n:5d}   {f:.3f}    {r:.3f}")

# 점수 일관성: 학습 데이터를 부트스트랩으로 10번 다시 뽑아 학습해도 같은 등급이 나오는 비율 + 불확실성(표준편차)
rng = np.random.default_rng(0)
boot = []
for b in range(10):
    idx = rng.integers(0, len(train), len(train))
    m = new_model().fit(train.iloc[idx][FEATS], train.iloc[idx]["ok"])
    boot.append(m.predict_proba(test[FEATS])[:, 1])
boot = np.array(boot)
same = np.mean([np.mean(grade(bp) == test["grade"].values) for bp in boot])
test["unc"] = boot.std(axis=0)
print(f"\n신뢰도 등급 일관성 {same:.1%} (목표 90% 이상: {'달성' if same >= 0.9 else '미달'}), 점수 불확실성(표준편차) 평균 {test['unc'].mean():.3f}, 최대 {test['unc'].max():.3f}")
print("[실제 링크 상태별 등급 분포]")
print(pd.crosstab(test["state"], test["grade"]).reindex(columns=["High", "Medium", "Low"]).to_string())
imp = permutation_importance(model, test[FEATS], test["ok"], scoring="roc_auc", n_repeats=3, random_state=0)
print("판단 근거(특징 중요도, AUC 감소량):", {f: round(float(v), 3) for f, v in sorted(zip(FEATS, imp.importances_mean), key=lambda x: -x[1])})
print("등급 운용 기준: High=정상 전송, Medium=긴급 전문 중복 전송 검토, Low=통신관에게 경로 변경·중계 검토 요청(자동 전환 아님)")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
fpr, tpr, _ = roc_curve(test["ok"], test["p"])
ax[0].plot(fpr, tpr); ax[0].plot([0, 1], [0, 1], "k--"); ax[0].set_title(f"ROC (AUC={auc:.3f})"); ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR")
seg = test.iloc[900:1200]
ax[1].plot(seg.index, seg["p"], label="success prob."); ax[1].fill_between(seg.index, seg["p"] - 2 * seg["unc"], seg["p"] + 2 * seg["unc"], alpha=0.3, label="+-2 sd")
ax[1].plot(seg.index, seg["p_true"], "k:", label="true prob."); ax[1].axhline(0.6, c="r", ls="--"); ax[1].legend(fontsize=8)
ax[1].set_title("Link reliability score with uncertainty (jamming at t=1000-1060)")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 05 예상 실행 결과 (Output)</b></summary>

```text
전송 성공 예측 ROC-AUC 0.891 (검증 창 5999개, 실제 불량 링크 창 293개)
                        방식   경보  오경보율  불량 재현율
기존 규칙(SNR<12dB 또는 손실>=40%) 1707 0.878   0.713
          신뢰도 엔진(성공확률<0.6)  363 0.342   0.816
오경보율 감축 61.0% (목표 40% 이상: 달성)

경보 임계값  경보수  오경보율  불량 재현율
   0.4        159   0.069    0.505
   0.5        220   0.150    0.638
   0.6        363   0.342    0.816
   0.7        709   0.626    0.904
   0.8       1232   0.768    0.976

신뢰도 등급 일관성 93.2% (목표 90% 이상: 달성), 점수 불확실성(표준편차) 평균 0.023, 최대 0.215
[실제 링크 상태별 등급 분포]
grade  High  Medium  Low
state                   
bad     333     965  128
good   4207     122   64
jam       1       8  171
판단 근거(특징 중요도, AUC 감소량): {'snr_db': 0.131, 'queue_util': 0.068, 'rtt_jitter': 0.045, 'pkt_bytes': 0.016, 'recent_loss': 0.016}
등급 운용 기준: High=정상 전송, Medium=긴급 전문 중복 전송 검토, Low=통신관에게 경로 변경·중계 검토 요청(자동 전환 아님)
```
</details>
